# 第 7 章 · 多 agent（Parallel Processing）

两个 agent **同时**跑，各写各的 state 键，最后合起来看。

和 Sequential 的唯一区别是并发：它们之间不能有数据依赖。

kernel 选 `Python (agentic-adk)`。

In [2]:
import sys
from pathlib import Path

# 【适配】定位仓库根；后端（官方 / 中转站）由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break

# 通用脚手架都在 common 里：run_once / run_session / run_async / 重试 / 静音
from common import (adk_model, quiet_adk_sync_warnings, retry_call,
                    run_once, run_session, run_async)
quiet_adk_sync_warnings()

try:
    import google.adk  # noqa: F401  —— 只为内核检查
except ModuleNotFoundError:
    raise RuntimeError(
        "本 notebook 需要内核『Python (agentic-adk)』，当前内核里没有 google.adk。\n"
        "VS Code 里点右上角内核名 → 选 Python (agentic-adk) → 重新运行本格。"
    ) from None

from google.adk.agents import LlmAgent, ParallelAgent

weather_fetcher = LlmAgent(
    name="weather_fetcher",
    model=adk_model(),                      # 【适配】gemini-2.0-flash-exp 已下线
    instruction="给出所给地点的天气，只回一句话。",
    output_key="weather_data",
)

news_fetcher = LlmAgent(
    name="news_fetcher",
    model=adk_model(),                      # 【适配】同上
    instruction="给出所给主题的一条新闻标题和一句话摘要。",
    output_key="news_data",
)

# ★ 本章新增 ParallelAgent：子 agent 同时启动，全部结束后才算自己结束
data_gatherer = ParallelAgent(name="data_gatherer", sub_agents=[weather_fetcher, news_fetcher])

finals, state = run_once(data_gatherer, "地点：北京；主题：大型户外运动活动")
for author, text in finals:
    print(f"\n--- [{author}] ---\n{text[:300]}")
print("\n--- 汇合后的 state ---")
print("weather_data:", (state.get("weather_data") or "(空)")[:200])
print("news_data   :", (state.get("news_data") or "(空)")[:200])


--- [weather_fetcher] ---
北京今天天气晴朗宜人，气温22℃，微风，空气质量优，非常适合开展大型户外运动活动。

--- [news_fetcher] ---
**新闻标题**：
2024北京金秋户外运动季启动，万余名运动爱好者齐聚温榆河畔

**一句话摘要**：
本次活动涵盖了山地骑行、徒步越野及户外露营等丰富项目，旨在倡导绿色健康的生活方式，进一步激发北京体育文旅消费的新活力。

--- 汇合后的 state ---
weather_data: 北京今天天气晴朗宜人，气温22℃，微风，空气质量优，非常适合开展大型户外运动活动。
news_data   : **新闻标题**：
2024北京金秋户外运动季启动，万余名运动爱好者齐聚温榆河畔

**一句话摘要**：
本次活动涵盖了山地骑行、徒步越野及户外露营等丰富项目，旨在倡导绿色健康的生活方式，进一步激发北京体育文旅消费的新活力。
